# LangGraph：重启、审批与重复写入

真实 SDK 1.2.14 + SQLite 插件 3.1.1；Python ≥3.10，CPU，无模型密钥。约 25–35 分钟。
先预测每个结果，再运行；检索和起草使用固定教学材料，不评价模型能力。
数据写入本次临时目录；重复运行会创建新目录，不触碰原记录。


In [ ]:
import os, sys, subprocess
assert sys.version_info >= (3, 10), '请使用 Python 3.10 或更高版本'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'langgraph==1.2.14', 'langgraph-checkpoint==4.2.0', 'langgraph-checkpoint-sqlite==3.1.1'])
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['LANGGRAPH_STRICT_MSGPACK'] = 'true'


## 1. 两个并行分支怎样合并状态？

两个分支在同一 super-step 写同一个普通字段会报错，不是“最后一个自动获胜”。
下面先运行失败基线，再用带稳定证据 ID 的 reducer 合并。冲突内容应拒绝；不要依赖完成顺序。


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.errors import InvalidUpdateError

class Plain(TypedDict):
    evidence: dict[str, str]

def merge_evidence(old, new):
    for key in old.keys() & new.keys():
        if old[key] != new[key]:
            raise ValueError('同一证据 ID 的内容发生冲突')
    return {**old, **new}

class Merged(TypedDict):
    evidence: Annotated[dict[str, str], merge_evidence]

def parallel(schema):
    b = StateGraph(schema)
    b.add_node('left', lambda s: {'evidence': {'a': '检查点'}})
    b.add_node('right', lambda s: {'evidence': {'b': '中断'}})
    b.add_edge(START, 'left'); b.add_edge(START, 'right')
    b.add_edge('left', END); b.add_edge('right', END)
    return b.compile()

try:
    parallel(Plain).invoke({'evidence': {}})
    raise AssertionError('失败基线没有暴露并发写入冲突')
except InvalidUpdateError:
    print('普通字段：InvalidUpdateError（预期）')
merged = parallel(Merged).invoke({'evidence': {}})['evidence']
assert merged == {'a': '检查点', 'b': '中断'}
assert merge_evidence(merged, {'a': '检查点'}) == merged
try:
    merge_evidence(merged, {'a': '不同内容'})
    raise AssertionError('没有拒绝冲突')
except ValueError:
    pass
print('ID 合并：', merged, '；相同证据去重，不同内容拒绝')


## 2. 构建检索 → 起草 → 审核 → 保存

每个节点返回局部状态更新。审核只接受明确布尔值，并将审批绑定草稿版本。
`checkpoints.sqlite` 保存图状态；`business.sqlite` 保存结果与观测计数。它们不是一笔共同事务。
阅读下面的完整 worker，重点找出 interrupt 前的计数、保存节点的幂等键，以及 commit 后的崩溃位置。


In [ ]:
import json, tempfile
from pathlib import Path
lab = Path(tempfile.mkdtemp(prefix='langgraph-study-'))
worker_source = r"""
import hashlib, json, os, sqlite3, sys
from pathlib import Path
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver

root, mode, thread = Path(sys.argv[1]), sys.argv[2], sys.argv[3]
config = {'configurable': {'thread_id': thread}}
business = root / 'business.sqlite'
with sqlite3.connect(business) as db:
    db.execute('CREATE TABLE IF NOT EXISTS results (op_id TEXT PRIMARY KEY, payload TEXT, receipt TEXT)')
    db.execute('CREATE TABLE IF NOT EXISTS counters (key TEXT PRIMARY KEY, n INTEGER)')

def bump(db, key):
    db.execute('INSERT INTO counters VALUES (?,1) ON CONFLICT(key) DO UPDATE SET n=n+1', (key,))

class State(TypedDict, total=False):
    task_id: str
    documents: list[str]
    draft: str
    draft_version: str
    approval: dict
    receipt: str
    status: str

def retrieve(state):
    # 固定教学材料；这里没有联网检索或模型调用。
    return {'documents': ['检查点保存图状态。', '中断恢复会重跑节点。']}

def draft(state):
    text = '\n'.join(state['documents'])
    return {'draft': text, 'draft_version': hashlib.sha256(text.encode()).hexdigest()[:12],
            'approval': {}, 'receipt': '', 'status': 'drafted'}

def approve(state):
    # 持久计数仅用于观察重入；它本身是审批前可重复的遥测写入。
    with sqlite3.connect(business) as db:
        bump(db, thread + ':before_interrupt')
    decision = interrupt({'draft': state['draft'], 'draft_version': state['draft_version']})
    if not isinstance(decision, dict) or type(decision.get('approved')) is not bool:
        raise ValueError('审批必须包含布尔 approved 与明确的版本')
    if decision.get('draft_version') != state['draft_version']:
        raise ValueError('审批版本与当前草稿不匹配')
    return {'approval': decision, 'status': 'approved' if decision['approved'] else 'rejected'}

def save(state):
    if state['approval'].get('approved') is not True or state['approval'].get('draft_version') != state['draft_version']:
        raise ValueError('缺少当前草稿的批准')
    if hashlib.sha256(state['draft'].encode()).hexdigest()[:12] != state['draft_version']:
        raise ValueError('草稿版本与内容不一致')
    operation_id = state['task_id'] + ':' + state['draft_version']
    payload = state['draft']
    with sqlite3.connect(business) as db:
        db.execute('BEGIN IMMEDIATE')
        old = db.execute('SELECT payload, receipt FROM results WHERE op_id=?', (operation_id,)).fetchone()
        if old and old[0] != payload:
            raise ValueError('相同操作 ID 不得复用不同参数')
        bump(db, operation_id + ':write_attempts')
        receipt = old[1] if old else 'receipt-' + hashlib.sha256(operation_id.encode()).hexdigest()[:12]
        if not old:
            db.execute('INSERT INTO results VALUES (?,?,?)', (operation_id, payload, receipt))
    # 业务库已 commit，但节点还没有返回，图的结果检查点尚未写入。
    if os.environ.get('CRASH_AFTER_WRITE') == '1':
        os._exit(17)
    return {'receipt': receipt, 'status': 'saved'}

def build(saver):
    builder = StateGraph(State)
    for name, fn in [('retrieve', retrieve), ('draft', draft), ('approve', approve), ('save', save)]:
        builder.add_node(name, fn)
    builder.add_edge(START, 'retrieve')
    builder.add_edge('retrieve', 'draft')
    builder.add_edge('draft', 'approve')
    builder.add_conditional_edges('approve', lambda s: 'save' if s['approval']['approved'] else END)
    builder.add_edge('save', END)
    return builder.compile(checkpointer=saver)

def run(graph):
    if mode == 'start':
        graph.invoke({'task_id': thread}, config, durability='sync')
    elif mode == 'resume':
        graph.invoke(Command(resume=json.loads(sys.argv[4])), config, durability='sync')
    elif mode == 'continue':
        graph.invoke(None, config, durability='sync')
    elif mode == 'fork':
        old = next(s for s in graph.get_state_history(config) if s.next == ('approve',))
        revised = old.values['draft'] + '\n新增的约束必须重新审批。'
        fork = graph.update_state(old.config, {'draft': revised,
            'draft_version': hashlib.sha256(revised.encode()).hexdigest()[:12],
            'approval': {}, 'receipt': ''}, as_node='draft')
        graph.invoke(None, fork, durability='sync')
        assert graph.get_state(old.config).values['draft'] == old.values['draft']
    elif mode == 'history':
        print(json.dumps([{'next': list(s.next), 'step': s.metadata.get('step'),
            'checkpoint_id': s.config['configurable']['checkpoint_id']} for s in graph.get_state_history(config)]))
        return
    snapshot = graph.get_state(config)
    print(json.dumps({'pid': os.getpid(), 'values': snapshot.values, 'next': list(snapshot.next)}, ensure_ascii=False))

if mode == 'memory-inspect':
    run(build(InMemorySaver()))
else:
    with SqliteSaver.from_conn_string(str(root / 'checkpoints.sqlite')) as saver:
        run(build(saver))

"""
worker_path = lab / 'workflow_worker.py'
worker_path.write_text(worker_source, encoding='utf-8')
print('实验目录：', lab)


## 3. 关闭进程以后真的能恢复吗？

每次 `run_worker` 都启动独立 Python 进程。暂停以后先检查状态，再注入业务写入后的硬退出。
预测：此时图还想执行哪个节点？业务已经写了几条？新进程继续后会调用几次保存，结果有几条？


In [ ]:
import os, sqlite3, sys, subprocess
def run_worker(mode, thread='approval-A', decision=None, crash=False, expected=0):
    args = [sys.executable, str(worker_path), str(lab), mode, thread]
    if decision is not None:
        args.append(json.dumps(decision))
    env = dict(os.environ, CRASH_AFTER_WRITE='1' if crash else '0')
    completed = subprocess.run(args, capture_output=True, text=True, env=env, timeout=30)
    assert completed.returncode == expected, completed.stderr
    return json.loads(completed.stdout) if completed.stdout.strip() else None

def metrics(thread, version):
    op = thread + ':' + version
    with sqlite3.connect(lab / 'business.sqlite') as db:
        rows = db.execute('SELECT COUNT(*) FROM results WHERE op_id=?', (op,)).fetchone()[0]
        def n(key):
            value = db.execute('SELECT n FROM counters WHERE key=?', (key,)).fetchone()
            return value[0] if value else 0
        return {'business_rows': rows, 'write_attempts': n(op+':write_attempts'),
                'before_interrupt': n(thread+':before_interrupt')}

paused = run_worker('start')
version = paused['values']['draft_version']
assert paused['next'] == ['approve']
assert metrics('approval-A', version) == {'business_rows':0, 'write_attempts':0, 'before_interrupt':1}
inspected = run_worker('inspect')
assert inspected['pid'] != paused['pid'] and inspected['values'] == paused['values']
assert run_worker('memory-inspect')['values'] == {}
assert run_worker('inspect', thread='unknown-thread')['values'] == {}

decision = {'approved': True, 'draft_version': version}
run_worker('resume', decision=decision, crash=True, expected=17)
after_crash = run_worker('inspect')
crash_metrics = metrics('approval-A', version)
assert after_crash['next'] == ['save'] and not after_crash['values']['receipt']
assert crash_metrics == {'business_rows':1, 'write_attempts':1, 'before_interrupt':2}

recovered = run_worker('continue')
recovered_metrics = metrics('approval-A', version)
assert recovered['next'] == [] and recovered['values']['receipt']
assert recovered_metrics == {'business_rows':1, 'write_attempts':2, 'before_interrupt':2}
run_worker('continue')  # 完成状态上继续没有待执行节点。
assert metrics('approval-A', version) == recovered_metrics
print('写后硬退出：', crash_metrics, 'next =', after_crash['next'])
print('新进程恢复：', recovered_metrics, 'next =', recovered['next'])


## 4. 拒绝、历史与修改草稿的分叉

拒绝应终止而不写结果。`get_state_history` 只读历史，不执行节点；从旧检查点分叉会重新到达审核。
新草稿必须取得新版本的批准，不能借用旧批准。旧检查点与已提交业务结果也不会被分叉撤销。


In [ ]:
rejection = run_worker('start', thread='approval-B')
rejected = run_worker('resume', thread='approval-B', decision={
    'approved': False, 'draft_version': rejection['values']['draft_version']})
rejected_metrics = metrics('approval-B', rejection['values']['draft_version'])
assert rejected['values']['status'] == 'rejected' and rejected['next'] == []
assert rejected_metrics['business_rows'] == rejected_metrics['write_attempts'] == 0

history = run_worker('history')
assert any(s['next'] == ['approve'] for s in history) and any(s['next'] == ['save'] for s in history)
forked = run_worker('fork')
assert forked['next'] == ['approve'] and forked['values']['draft_version'] != version
run_worker('resume', decision=decision, expected=1)  # 明确拒绝旧草稿的审批。
assert metrics('approval-A', version)['business_rows'] == 1
assert metrics('approval-A', forked['values']['draft_version'])['business_rows'] == 0
print('拒绝分支：', rejected_metrics)
print('历史检查点：', len(history), '；新草稿审批待处理；旧批准被拒绝。')


## 5. 保存观测与自己的解释

将下面报告导入网站的“导入实验结果”。先写自己的预测，再比较真实结果。
本例检验同机本地 SQLite、真实进程退出和确定性节点，不检验远端服务、多副本、模型输出或硬盘掉电。
可继续将 `approved` 改为字符串、换 `thread_id` 或删除幂等查询，在新的实验目录比较结果。


In [ ]:
from importlib.metadata import version as package_version
report = {
    'experiment': 'langgraph-durable-workflow',
    'data_scope': 'local SDK + SQLite subprocess experiment',
    'model_called': False,
    'versions': {name: package_version(name) for name in
        ['langgraph', 'langgraph-checkpoint', 'langgraph-checkpoint-sqlite']},
    'paused': {'next': paused['next'], 'before_interrupt': 1},
    'after_crash': {'next': after_crash['next'], **crash_metrics},
    'recovered': {'next': recovered['next'], **recovered_metrics},
    'rejected': rejected_metrics,
    'thread_isolated': True, 'new_memory_empty': True,
    'fork_requires_new_approval': True, 'reducer_keys': sorted(merged),
    'my_explanation': ''
}
Path('langgraph_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))


## 原始依据

[Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) ·
[Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers) ·
[Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) ·
[Time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel)

以上是 SDK 机制依据；本笔记本的教学材料、故障点和断言为本站原创实验。SQLite 文件默认保留在本次临时目录，便于检查；确认报告后可删除该目录。
